<a href="https://colab.research.google.com/github/saids193gmail/ERP-Synchronisation-Error-Analysis/blob/main/ERP_Synchronisation_Analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ERP Synchronisation Error Analysis

This notebook contains the Python analysis workflow used for the study:

**“SynchIntegrity: A Data-Driven Framework for ERP Synchronisation Failure Analysis and Prevention in Higher Education”**

## Data availability and confidentiality

The research dataset is not included in this repository because it contains
confidential institutional records and is subject to institutional data-protection
requirements.

Access to the underlying data may be provided by the corresponding author upon
reasonable request and subject to institutional approval.

This notebook is published to support transparency and reproducibility of the
analytical methodology. It reproduces the analysis pipeline for Figures 2–5
when an authorised copy of the research dataset is supplied.

**Important:** Do not commit the research dataset or executed notebook outputs
containing record-level data to a public repository.


## 1. Required libraries


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter, PercentFormatter
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import PCA
from mpl_toolkits.axes_grid1.inset_locator import inset_axes, mark_inset

print("Libraries loaded successfully.")


## 2. Load the authorised research dataset

The CSV file is intentionally excluded from the public repository. In Google Colab, an authorised user can upload the private dataset for analysis. The expected public-facing filename is `research_dataset.csv`.


In [ ]:
from google.colab import files

uploaded = files.upload()

In [ ]:
# The research dataset is not included in this repository due to
# institutional confidentiality and data protection requirements.
# Authorized users may upload the dataset using the cell above.

df = pd.read_csv("research_dataset.csv")

print("Rows:", len(df))
print("Columns:", len(df.columns))


## 3. Data preprocessing


In [ ]:
df.columns = (
    df.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "", regex=False)
)

print(df.columns.tolist())

In [ ]:
# Clean error descriptions
df["error_description"] = (
    df["error_description"]
    .astype(str)
    .str.lower()
    .str.strip()
)

# Convert date to datetime
df["timestamp"] = pd.to_datetime(
    df["date"],
    errors="coerce"
)

# Extract reporting year
df["year"] = df["timestamp"].dt.year

# Check results
print("Missing dates:", df["timestamp"].isna().sum())
print()
print("Records by year:")
print(df["year"].value_counts().sort_index())

## 4. Construct distinct error cases

A distinct error case is defined as the combination of **student identifier + normalised error description**. The identifier is used only for computation; record-level identifiers are not printed by this public notebook.


In [ ]:
# Create distinct error case identifier
# Distinct case = Student ID + Error Description

df["error_case"] = (
    df["studentid"].astype(str).str.strip()
    + " | "
    + df["error_description"].astype(str).str.strip()
)

print("Error case created successfully.")


## 5. Error classification

Error descriptions are mapped to the study categories using the documented keyword rules below. This code represents the automated classification component of the methodology; any manual review/validation described in the manuscript remains part of the research procedure.


In [ ]:
def classify_error_advanced(text):
    t = str(text).lower()

    # Cross-Institution / Registry Conflicts
    if any(k in t for k in [
        "another academic institution",
        "another institution",
        "acr",
        "same academic status",
        "same enrolled status",
        "recorded in asas",
        "not transferred by your system"
    ]):
        return "Cross-Institution / Registry Conflict"

    # Identity & Uniqueness
    if any(k in t for k in [
        "passport",
        "civil id",
        "civil number",
        "rop",
        "duplicate",
        "numeric",
        "spaces"
    ]):
        return "Identity & Uniqueness"

    # Metadata & Code Lists
    if any(k in t for k in [
        "major",
        "subject",
        "qualification",
        "qlfn",
        "nationality",
        "classification",
        "program",
        "accomodation",
        "funding"
    ]):
        return "Metadata & Code Lists"

    # Temporal & Academic Status Logic
    if any(k in t for k in [
        "graduated",
        "foundation year",
        "expected graduation",
        "start date",
        "sts_change",
        "year",
        "birth date"
    ]):
        return "Temporal & Status Logic"

    # Mandatory Field Completeness
    if any(k in t for k in [
        "empty",
        "mandatory",
        "required",
        "cannot be null",
        "not available"
    ]):
        return "Mandatory Field Completeness"

    # External Systems / Integration Failures
    if any(k in t for k in [
        "asas",
        "moheri",
        "connection error",
        "batch",
        "submission"
    ]):
        return "External System / Workflow Integration"

    # Numeric / System Error Codes
    if t.strip().isdigit():
        return "System / Unknown Error Code"

    return "Unclassified"


df["error_category"] = (
    df["error_description"]
    .apply(classify_error_advanced)
)

## 6. Figure 2 — Pareto analysis of ERP-to-regulatory synchronisation errors by category

Category totals are calculated directly from the classified research records rather than being manually entered.


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter, PercentFormatter

# ============================================================
# 1. DERIVE CATEGORY COUNTS FROM THE CLASSIFIED DATA
# ============================================================

pareto = (
    df["error_category"]
    .value_counts()
    .rename_axis("Error Category")
    .reset_index(name="Errors")
    .sort_values("Errors", ascending=False)
    .reset_index(drop=True)
)

# ============================================================
# 2. CALCULATE PERCENTAGES
# ============================================================

pareto["Percentage"] = (
    pareto["Errors"] /
    pareto["Errors"].sum()
) * 100

pareto["Cumulative %"] = (
    pareto["Percentage"].cumsum()
)

print(pareto)

# ============================================================
# 3. CREATE FIGURE
# ============================================================

fig, ax1 = plt.subplots(figsize=(9, 5))

# ============================================================
# 4. BAR CHART - NUMBER OF ERRORS
# ============================================================

bars = ax1.bar(
    pareto["Error Category"],
    pareto["Errors"],
    width=0.55,
    label="Errors"
)

ax1.set_ylabel(
    "Number of Errors",
    fontsize=10
)

# Format numbers with commas
ax1.yaxis.set_major_formatter(
    FuncFormatter(
        lambda x, pos: f"{int(x):,}"
    )
)

# X-axis labels
ax1.tick_params(
    axis="x",
    labelsize=8
)

plt.setp(
    ax1.get_xticklabels(),
    rotation=45,
    ha="right",
    rotation_mode="anchor"
)

# ============================================================
# 5. GRID LINES
# ============================================================

ax1.grid(
    axis="y",
    linestyle="--",
    linewidth=0.5,
    alpha=0.35
)

ax1.set_axisbelow(True)

# ============================================================
# 6. SECOND Y-AXIS - CUMULATIVE %
# ============================================================

ax2 = ax1.twinx()

line = ax2.plot(
    pareto["Error Category"],
    pareto["Cumulative %"],
    marker="o",
    linewidth=2,
    markersize=4,
    label="Cumulative %"
)

ax2.set_ylabel(
    "Cumulative Percentage",
    fontsize=10
)

ax2.set_ylim(0, 105)

ax2.set_yticks(
    range(0, 101, 10)
)

ax2.yaxis.set_major_formatter(
    PercentFormatter(xmax=100)
)

# ============================================================
# 7. OPTIONAL 80% PARETO REFERENCE LINE
# ============================================================

ax2.axhline(
    y=80,
    linestyle="--",
    linewidth=0.8,
    alpha=0.5
)

# ============================================================
# 8. FOUR-SIDED BORDER
# ============================================================

# Left, bottom and top borders from primary axis
for side in ["left", "bottom", "top"]:
    ax1.spines[side].set_visible(True)
    ax1.spines[side].set_linewidth(1.0)

# Right border from secondary axis
ax2.spines["right"].set_visible(True)
ax2.spines["right"].set_linewidth(1.0)

# Avoid duplicate top border
ax2.spines["top"].set_visible(False)

# ============================================================
# 9. LEGEND - INSIDE RIGHT SIDE
# ============================================================

handles1, labels1 = ax1.get_legend_handles_labels()
handles2, labels2 = ax2.get_legend_handles_labels()

ax1.legend(
    handles1 + handles2,
    labels1 + labels2,
    loc="center right",
    frameon=False,
    fontsize=8
)

# ============================================================
# 10. FINAL FORMATTING
# ============================================================

plt.tight_layout()

# ============================================================
# 11. SAVE HIGH-RESOLUTION IMAGE
# ============================================================

plt.savefig(
    "Figure_2_Pareto_ERP_Synchronisation_Errors.png",
    dpi=600,
    bbox_inches="tight"
)

plt.show()


## 7. Figure 3 — PCA visualisation of institutional and external ERP synchronisation error patterns

The error categories are first grouped into institutional versus external/registry-dependent errors. Unique error descriptions are represented with TF-IDF and reduced to two principal components for visualisation.


In [ ]:
external_categories = [
    "Cross-Institution / Registry Conflict",
    "External System / Workflow Integration"
]

df["error_cluster"] = np.where(
    df["error_category"].isin(external_categories),
    "External / Registry Error Cluster",
    "Institutional Error Cluster"
)

print(df["error_cluster"].value_counts())

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt

# One row per unique error description
plot_df = (
    df[
        ["error_description", "error_cluster"]
    ]
    .drop_duplicates("error_description")
    .reset_index(drop=True)
)

# Convert error descriptions to TF-IDF
vectorizer = TfidfVectorizer(
    stop_words="english",
    max_features=1000,
    ngram_range=(1, 2)
)

X = vectorizer.fit_transform(
    plot_df["error_description"]
)

# Reduce TF-IDF representation to two dimensions
pca = PCA(
    n_components=2,
    random_state=42
)

coords = pca.fit_transform(X.toarray())

plot_df["PC1"] = coords[:, 0]
plot_df["PC2"] = coords[:, 1]

print(plot_df["error_cluster"].value_counts())

In [ ]:
import matplotlib.pyplot as plt
from mpl_toolkits.axes_grid1.inset_locator import inset_axes, mark_inset

# ============================================================
# EXPLAINED VARIANCE
# ============================================================

explained_variance = pca.explained_variance_ratio_ * 100

print(f"PC1 explained variance: {explained_variance[0]:.2f}%")
print(f"PC2 explained variance: {explained_variance[1]:.2f}%")
print(f"Total explained variance: {explained_variance.sum():.2f}%")

# ============================================================
# CREATE MAIN FIGURE
# ============================================================

fig, ax = plt.subplots(figsize=(8, 5))

# ============================================================
# GROUP DEFINITIONS
# ============================================================

groups = [
    "Institutional Error Cluster",
    "External / Registry Error Cluster"
]

# Better labels for the paper
display_labels = {
    "Institutional Error Cluster":
        "Institutional errors",

    "External / Registry Error Cluster":
        "External / registry-dependent errors"
}

# ============================================================
# MAIN PCA SCATTER PLOT
# ============================================================

for group in groups:

    subset = plot_df[
        plot_df["error_cluster"] == group
    ]

    ax.scatter(
        subset["PC1"],
        subset["PC2"],
        s=38,
        alpha=0.80,
        label=display_labels[group]
    )

# ============================================================
# AXIS LABELS WITH EXPLAINED VARIANCE
# ============================================================

ax.set_xlabel(
    f"Principal Component 1 (PC1) "
    f"({explained_variance[0]:.1f}% variance)",
    fontsize=9
)

ax.set_ylabel(
    f"Principal Component 2 (PC2) "
    f"({explained_variance[1]:.1f}% variance)",
    fontsize=9
)

# ============================================================
# GRID
# ============================================================

ax.grid(
    linestyle="--",
    linewidth=0.4,
    alpha=0.30
)

ax.set_axisbelow(True)

# ============================================================
# FOUR-SIDED BORDER
# ============================================================

for side in [
    "left",
    "right",
    "top",
    "bottom"
]:
    ax.spines[side].set_visible(True)
    ax.spines[side].set_linewidth(1.0)

# ============================================================
# LEGEND
# ============================================================

ax.legend(
    loc="upper right",
    frameon=True,
    fontsize=8
)

# ============================================================
# ZOOMED INSET
# ============================================================

# Position of inset inside main figure
axins = inset_axes(
    ax,
    width="42%",
    height="42%",
    loc="center right",
    borderpad=1.8
)

# Plot same data inside inset
for group in groups:

    subset = plot_df[
        plot_df["error_cluster"] == group
    ]

    axins.scatter(
        subset["PC1"],
        subset["PC2"],
        s=25,
        alpha=0.80
    )

# ============================================================
# ZOOM REGION
# ============================================================

# These limits focus on the dense lower-left area.
# Adjust slightly if your data changes.

axins.set_xlim(
    -0.35,
    -0.22
)

axins.set_ylim(
    -0.23,
    0.03
)

# ============================================================
# INSET FORMATTING
# ============================================================

axins.grid(
    linestyle="--",
    linewidth=0.3,
    alpha=0.25
)

axins.tick_params(
    axis="both",
    labelsize=6
)

# Four borders around inset
for side in [
    "left",
    "right",
    "top",
    "bottom"
]:
    axins.spines[side].set_visible(True)
    axins.spines[side].set_linewidth(0.8)

# Small inset title
axins.set_title(
    "Zoomed region",
    fontsize=7
)

# ============================================================
# SHOW LOCATION OF ZOOMED REGION
# ============================================================

mark_inset(
    ax,
    axins,
    loc1=2,
    loc2=4,
    fc="none",
    ec="0.45",
    linewidth=0.7
)

# ============================================================
# TICK FORMATTING
# ============================================================

ax.tick_params(
    axis="both",
    labelsize=8
)

# ============================================================
# FINAL LAYOUT
# ============================================================

plt.tight_layout()

# ============================================================
# SAVE HIGH-RESOLUTION FIGURE
# ============================================================

plt.savefig(
    "Figure_3_PCA_Error_Patterns_with_Zoom.png",
    dpi=600,
    bbox_inches="tight"
)

plt.savefig(
    "Figure_3_PCA_Error_Patterns_with_Zoom.pdf",
    bbox_inches="tight"
)

plt.show()

## 8. Annual recurrence analysis

For each reporting year, total error events (Eₜ) and distinct error cases (Dₜ) are calculated. The recurrence ratio is defined as **((Eₜ − Dₜ) / Eₜ) × 100**.


In [ ]:
# ============================================================
# Annual Recurrence Analysis
# RR_t = ((E_t - D_t) / E_t) × 100
# ============================================================

# Remove rows without a valid year for this analysis
annual_df = df.dropna(subset=["year"]).copy()

# E_t = total error events per year
total_events = annual_df.groupby("year").size()

# D_t = distinct Student + Error Description cases per year
distinct_cases = (
    annual_df
    .groupby("year")["error_case"]
    .nunique()
)

# Build summary table
recurrence = pd.DataFrame({
    "Total Error Events (Et)": total_events,
    "Distinct Error Cases (Dt)": distinct_cases
}).reset_index()

# Repeated events
recurrence["Repeated Events (Et-Dt)"] = (
    recurrence["Total Error Events (Et)"]
    - recurrence["Distinct Error Cases (Dt)"]
)

# Equation (4)
recurrence["Recurrence Ratio (%)"] = (
    recurrence["Repeated Events (Et-Dt)"]
    / recurrence["Total Error Events (Et)"]
) * 100

# Format year
recurrence["year"] = recurrence["year"].astype(int)

# Keep 2017–2025
recurrence = recurrence[
    recurrence["year"].between(2017, 2025)
].sort_values("year")

# Round only for display
recurrence_display = recurrence.copy()
recurrence_display["Recurrence Ratio (%)"] = (
    recurrence_display["Recurrence Ratio (%)"].round(2)
)

print(recurrence_display.to_string(index=False))

## 9. Figure 4 — Annual total and distinct ERP-to-regulatory synchronisation error events (2017–2025)

The six-panel figure is generated from the annual recurrence table calculated above; annual totals are not manually entered.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter

# ============================================================
# 1. DERIVE ANNUAL DATA FROM THE RECURRENCE ANALYSIS
# ============================================================

df_chart = recurrence.rename(columns={
    "year": "Year",
    "Total Error Events (Et)": "Total Error Events",
    "Distinct Error Cases (Dt)": "Distinct Error Cases"
})[["Year", "Total Error Events", "Distinct Error Cases"]].copy()

# ============================================================
# 2. DERIVED MEASURES
# ============================================================

# Multiplicity:
# Number of recorded events per distinct error case
df_chart["Multiplicity Factor"] = (
    df_chart["Total Error Events"]
    / df_chart["Distinct Error Cases"]
)

# Repeated events
df_chart["Repeated Error Events"] = (
    df_chart["Total Error Events"]
    - df_chart["Distinct Error Cases"]
)

# Year-to-year percentage change
df_chart["YoY Change (%)"] = (
    df_chart["Total Error Events"]
    .pct_change() * 100
)

print(df_chart)

# ============================================================
# 3. CREATE 2 x 3 MULTI-PANEL FIGURE
# ============================================================

fig, axes = plt.subplots(
    2,
    3,
    figsize=(14, 8)
)

years = df_chart["Year"]

# ============================================================
# (a) TOTAL ERROR EVENTS
# ============================================================

ax = axes[0, 0]

ax.bar(
    years,
    df_chart["Total Error Events"]
)

ax.set_title(
    "(a) Total Error Events",
    fontsize=10
)

ax.set_ylabel("Number of Errors")

ax.yaxis.set_major_formatter(
    FuncFormatter(lambda x, pos: f"{int(x):,}")
)

# ============================================================
# (b) DISTINCT ERROR CASES
# ============================================================

ax = axes[0, 1]

ax.bar(
    years,
    df_chart["Distinct Error Cases"]
)

ax.set_title(
    "(b) Distinct Error Cases",
    fontsize=10
)

ax.set_ylabel("Number of Cases")

ax.yaxis.set_major_formatter(
    FuncFormatter(lambda x, pos: f"{int(x):,}")
)

# ============================================================
# (c) ERROR MULTIPLICITY
# ============================================================

ax = axes[0, 2]

ax.plot(
    years,
    df_chart["Multiplicity Factor"],
    marker="o",
    linewidth=1.8
)

ax.set_title(
    "(c) Error Multiplicity",
    fontsize=10
)

ax.set_ylabel("Events per Distinct Case")

# Add multiplicity labels
for year, factor in zip(
    years,
    df_chart["Multiplicity Factor"]
):
    ax.annotate(
        f"{factor:.1f}×",
        (year, factor),
        xytext=(0, 7),
        textcoords="offset points",
        ha="center",
        fontsize=7
    )

# ============================================================
# (d) TOTAL VS DISTINCT ERROR EVENTS
# ============================================================

ax = axes[1, 0]

ax.plot(
    years,
    df_chart["Total Error Events"],
    marker="o",
    linewidth=1.8,
    label="Total Error Events"
)

ax.plot(
    years,
    df_chart["Distinct Error Cases"],
    marker="o",
    linewidth=1.8,
    label="Distinct Error Cases"
)

ax.set_title(
    "(d) Total vs Distinct Error Events",
    fontsize=10
)

ax.set_ylabel("Number of Errors")

ax.yaxis.set_major_formatter(
    FuncFormatter(lambda x, pos: f"{int(x):,}")
)

ax.legend(
    fontsize=7,
    loc="upper left",
    frameon=True
)

# ============================================================
# (e) REPEATED ERROR EVENTS
# ============================================================

ax = axes[1, 1]

ax.bar(
    years,
    df_chart["Repeated Error Events"]
)

ax.set_title(
    "(e) Repeated Error Events",
    fontsize=10
)

ax.set_ylabel("Number of Repeated Events")

ax.yaxis.set_major_formatter(
    FuncFormatter(lambda x, pos: f"{int(x):,}")
)

# ============================================================
# (f) YEAR-TO-YEAR CHANGE
# ============================================================

ax = axes[1, 2]

# 2017 has no previous year, so it is excluded
yoy_data = df_chart.dropna(
    subset=["YoY Change (%)"]
)

ax.bar(
    yoy_data["Year"],
    yoy_data["YoY Change (%)"]
)

# Zero reference line
ax.axhline(
    y=0,
    linewidth=0.8
)

ax.set_title(
    "(f) Year-to-Year Change in Error Volume",
    fontsize=10
)

ax.set_ylabel("Change (%)")

# Add percentage labels
for year, value in zip(
    yoy_data["Year"],
    yoy_data["YoY Change (%)"]
):

    offset = 5 if value >= 0 else -12

    ax.annotate(
        f"{value:+.1f}%",
        (year, value),
        xytext=(0, offset),
        textcoords="offset points",
        ha="center",
        fontsize=7
    )

# ============================================================
# 4. COMMON FORMATTING
# ============================================================

for ax in axes.flat:

    ax.set_xlabel("Year")

    ax.set_xticks(years)

    ax.tick_params(
        axis="x",
        rotation=45,
        labelsize=7
    )

    ax.tick_params(
        axis="y",
        labelsize=8
    )

    # Grid
    ax.grid(
        axis="y",
        linestyle="--",
        linewidth=0.4,
        alpha=0.3
    )

    ax.set_axisbelow(True)

    # Four borders
    for side in [
        "left",
        "right",
        "top",
        "bottom"
    ]:
        ax.spines[side].set_visible(True)
        ax.spines[side].set_linewidth(0.8)

# ============================================================
# 5. LAYOUT
# ============================================================

plt.tight_layout(
    h_pad=2.0,
    w_pad=1.5
)

# ============================================================
# 6. SAVE HIGH-RESOLUTION FIGURE
# ============================================================

plt.savefig(
    "Figure_4_Six_Panel_Annual_Error_Analysis.png",
    dpi=600,
    bbox_inches="tight"
)

plt.show()


## 10. Figure 5 — Annual ERP-to-regulatory synchronisation error volume and recurrence ratio (2017–2025)

This final figure combines annual total error volume with the recurrence ratio calculated from the same annual recurrence table.


In [ ]:
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter

fig, ax1 = plt.subplots(figsize=(9, 4.8))

years = recurrence["year"]
events = recurrence["Total Error Events (Et)"]
ratios = recurrence["Recurrence Ratio (%)"]

# ============================================================
# BAR: Total error events
# ============================================================

bars = ax1.bar(
    years,
    events,
    width=0.60,
    alpha=0.45,
    label="Total Error Events"
)

ax1.set_xlabel(
    "Year",
    fontsize=10
)

ax1.set_ylabel(
    "Total Error Events",
    fontsize=10
)

ax1.yaxis.set_major_formatter(
    FuncFormatter(
        lambda x, pos:
        f"{int(x/1000)}k"
        if x >= 1000
        else f"{int(x)}"
    )
)

ax1.set_xticks(years)

# ============================================================
# LINE: Recurrence ratio
# ============================================================

ax2 = ax1.twinx()

line = ax2.plot(
    years,
    ratios,
    marker="o",
    linewidth=2.2,
    markersize=6,
    label="Recurrence Ratio"
)

ax2.set_ylabel(
    "Recurrence Ratio (%)",
    fontsize=10
)

ax2.set_ylim(60, 100)

ax2.set_yticks([
    60,
    70,
    80,
    90,
    100
])

# ============================================================
# PERCENTAGE LABELS
# ============================================================

for year, ratio in zip(
    years,
    ratios
):
    ax2.annotate(
        f"{ratio:.1f}%",
        (year, ratio),
        xytext=(0, 8),
        textcoords="offset points",
        ha="center",
        fontsize=8
    )

# ============================================================
# GRID
# ============================================================

ax1.grid(
    axis="y",
    linestyle="--",
    linewidth=0.5,
    alpha=0.4
)

ax1.set_axisbelow(True)

# ============================================================
# FOUR-SIDED BOX / BORDER
# ============================================================

# Left, bottom and top borders from ax1
for side in [
    "left",
    "bottom",
    "top"
]:
    ax1.spines[side].set_visible(True)
    ax1.spines[side].set_linewidth(1.0)

# Right border from ax2
ax2.spines["right"].set_visible(True)
ax2.spines["right"].set_linewidth(1.0)

# Avoid duplicate borders from ax2
ax2.spines["top"].set_visible(False)
ax2.spines["left"].set_visible(False)
ax2.spines["bottom"].set_visible(False)

# ============================================================
# LEGEND
# ============================================================

handles1, labels1 = ax1.get_legend_handles_labels()
handles2, labels2 = ax2.get_legend_handles_labels()

ax1.legend(
    handles1 + handles2,
    labels1 + labels2,
    loc="upper left",
    frameon=False,
    fontsize=8
)

# ============================================================
# TICK FORMATTING
# ============================================================

ax1.tick_params(
    axis="both",
    labelsize=9
)

ax2.tick_params(
    axis="y",
    labelsize=9
)

# ============================================================
# FINAL LAYOUT
# ============================================================

plt.tight_layout()

# ============================================================
# SAVE
# ============================================================

plt.savefig(
    "Figure_5_Recurrence_and_Error_Volume.png",
    dpi=600,
    bbox_inches="tight"
)

plt.savefig(
    "Figure_5_Recurrence_and_Error_Volume.pdf",
    bbox_inches="tight"
)

plt.show()

## Reproducibility note

The public repository intentionally excludes the confidential dataset and saved record-level outputs. With an authorised dataset having the required schema, the notebook is organised to run sequentially from preprocessing through the four figures reported in the manuscript.
